# **WGCNA-in-Python**

Weighted correlation network analysis, also known as weighted gene co-expression network analysis (WGCNA), is a widely used data mining method especially for studying biological networks based on pairwise correlations between variables.

## **Pipeline notebook**

This notebook provides a customizable and efficient guide through the attached program ([here_at_this_github_repository](https://github.com/riccardocaccia/WGCNA-in-python/blob/main/WGCNA_1.2.py)) , useful for constructing gene co-expression networks from expression data (CSV/TSV), performing correlation analysis, clustering, network construction, centrality analysis, and optional GO enrichment and Cytoscape export.

## **Features**
---

- Load gene expression matrix (`.csv` or `.tsv`, also unzip if necessary)
- Filter genes by expression threshold and variance
- Parallelized correlation matrix calculation (Pearson, Spearman, Kendall)
- Plot interactive heatmaps and dendrograms
- Construct gene co-expression network with user-defined correlation threshold
- Analyze network centrality metrics
- Save `.graphml` file for Cytoscape visualization
- Perform Gene Ontology (GO) enrichment for detected communities (internet connection needed)

---

## Requirements

Install dependencies via `pip`:

```bash
pip install polars pandas numpy matplotlib seaborn networkx scipy gseapy plotly
```
## Author
[Riccardo Caccia](https://github.com/riccardocaccia) <br>
riccardo2.caccia@mail.polimi.it <br>
10742136-271218 

### libraries needed

In [ ]:
import argparse #parser
import os #used for path and creating output dirs
import numpy as np
import polars as pl
import pandas as pd #manage better big datafrme
import matplotlib.pyplot as plt
import seaborn as sns 
import networkx as nx #building networks
from networkx.algorithms import community
from scipy.cluster.hierarchy import linkage, dendrogram #obain the dendrogram and calculate the linkage
from scipy.spatial.distance import squareform
import gseapy as gp #used for GO enrichment
import multiprocessing as mp #multiprocessing for big dataset
import plotly.express as px #interactive plot
import plotly.figure_factory as ff
import zipfile #manage zip file

### Useful class
 - `Custom Error Class`: A class is created to raise a personalized exception in case of an invalid model selection. This is especially useful for handling incorrect calls in the command-line interface.

- `AbsolutePath Class`: This class creates, for each input file you wish to analyze (e.g., [file1, file2, file3]), a dedicated output folder named after the input file. This helps clearly organize and identify the results once your automated pipeline has completed processing.

In [ ]:
class MethodError(Exception):
    def __init__(self, method):
        self.method = method
        message = f"Invalid method: '{method}'. Choose 'pearson' | 'spearman' | 'kendall'."
        super().__init__(message)

class AbsolutePath:
    def __init__(self, basename):
        self.basename = basename
        self.output_dir = f'{basename}_WGCNA_output'
        os.makedirs(self.output_dir, exist_ok=True)

### Loading File

This function receive as input a **MicroArray result** (an expression table, eith genes on rows, samples and expression levels on the columns), it is possible to upload a `.tsv`, a `.csv` and also a `zipped file`. <br>

**Parameters**
- `expression_file` : A MicroArray expression marix.
- `basename_obj` (`AbsolutePath`): is used to build the name of the output dir and the other results
- `output_dir`: as above, is used to store the results in a dedicated directory

**returns**
- A polars Data Frame

In [ ]:
def load_expression_data(expression_file, basename_obj: AbsolutePath, output_dir):
    ext = os.path.splitext(expression_file)[1].lower()

    if ext == '.zip':
        with zipfile.ZipFile(expression_file, 'r') as zipf:
            zipf.extractall(os.path.join(output_dir, f'{basename_obj.basename}'))
            extracted_files = zipf.namelist()

            if not extracted_files:
                raise ValueError('zip file is empty')
            
            expression_file = os.path.join(output_dir, f'{basename_obj.basename}', extracted_files[0])
            ext = os.path.splitext(expression_file)[1].lower()

    match ext:
        case '.tsv':
            df = pl.read_csv(expression_file, separator='\t')
        case '.csv':
            df = pl.read_csv(expression_file)
        case _:
            raise ValueError(f'Unsupported file format: {ext}')
    return df

### Subsetting genes

This function filters a gene expression dataset to retain only the top n genes with the highest variance across samples. High variance genes are often more informative in downstream analyses such as clustering or dimensionality reduction.

**Behavior**

1. Checks whether the input DataFrame is empty. Raises an error if so.
2. Computes the mean and sum of squares across all numeric (expression) columns for each gene.
3. Calculates the variance using the formula:  
   $$
   \text{Var} = \frac{\sum x^2}{n} - (\text{mean})^2
   $$
4. Sorts the genes by variance in descending order and selects the top `n`.
5. If no genes remain after filtering, raises an error.
6. Returns a cleaned DataFrame containing only the selected genes and original columns, with temporary computation columns removed.

**Returns**

- A Polars DataFrame of the top `n` most variable genes, without intermediate computation columns.

In [ ]:
def subset_high_variance_genes(df: pl.DataFrame, top_n: int):
    if df.shape[0] == 0:
        raise ValueError('no genes after filtering.')
    
    numeric_cols = df.columns[1:]

    df = df.with_columns([                                                    
        pl.mean_horizontal([pl.col(c) for c in numeric_cols]).alias('mean_expr'),
        pl.sum_horizontal([(pl.col(c) ** 2) for c in numeric_cols]).alias('sum_squares')
    ])

    n = len(numeric_cols)

    df = df.with_columns(
        ((pl.col('sum_squares') / n) - (pl.col('mean_expr') ** 2)).alias('row_var') 
    )
    top_genes = df.sort('row_var', descending=True).head(top_n)

    if top_genes.is_empty():
        raise ValueError("No genes selected after variance filtering.")
    
    return top_genes.drop(["mean_expr", "sum_squares", "row_var"])

### Removing least relevant genes

this function fill the empty cell with an arbitrary 0, to not falsificate the average, and discard all the genes with a mean between the cell less than a given threshold (default = 0.85)

**Parameters**
- `expression_dataset` (`pl.DataFrame`): now that i loaded, we pass as input the expression matrix
- `expression_threshold`: an optional threshold to discriminate between genes expression

**Behavior**

1. Fill the empty cells with an arbitrary 0, to not fake the results.
2. Discriminate between the genes with an average between the cells less than a given threshold (default=0.85).

**Returns**

- Returns a cleaned DataFrame containing only the selected genes, the most expressed one.

In [ ]:
def remove_least_expressed_genes(expression_dataset: pl.DataFrame, expression_threshold: float):
    expr_no_na = expression_dataset.fill_null(0)
    numeric_cols = expr_no_na.columns[1:]
    expr_with_mean = expr_no_na.with_columns(
        pl.mean_horizontal([pl.col(c) for c in numeric_cols]).alias("mean_expr")
    )
    filtered = expr_with_mean.filter(pl.col("mean_expr") >= expression_threshold)
    return filtered.drop("mean_expr")

### Parallel Correlation

This function computes a pairwise correlation matrix between genes (rows) in parallel, using multiprocessing and a private function to speed up computation on large datasets.

**Parameters**
- `df` (`pd.DataFrame`): A pandas DataFrame where each row represents a gene and columns represent expression values. The first column is assumed to contain gene identifiers and is used as the index.
- `method` (`str`): The correlation method to use. Options:
  - `'pearson'`: Pearson correlation coefficient
  - `'spearman'`: Spearman rank correlation
  - `'kendall'`: Kendall tau correlation
- `n_processes` (`int`): Number of parallel processes to use. If set to `0` or `None`, all available CPU cores will be used.

**Behavior**
1. Sets the first column of the DataFrame as the index (assumed to be gene names).
2. Converts the expression matrix to a NumPy array.
3. Splits the data into chunks based on the number of processes.
4. Each chunk is processed by the `_correlate_chunk` function, which computes correlation row by row.
5. Correlation matrices from each chunk are concatenated into a final square matrix.
6. The resulting correlation matrix is returned as a `DataFrame`, with rounding and removal of any rows/columns entirely filled with `NaN`.

**Returns**
- `pd.DataFrame`: A symmetric correlation matrix of shape `(n_genes, n_genes)`.

---

### `_correlate_chunk(args)`

**Private helper function** used internally by `parallel_correlation` to compute row-wise correlations for a subset (chunk) of the expression matrix.

#### **Parameters**
- `args` (`tuple`): A tuple containing:
  - `data` (`np.ndarray`): Full expression matrix.
  - `method` (`str`): Correlation method.
  - `start` (`int`): Start index of the chunk.
  - `end` (`int`): End index of the chunk.

#### **Returns**
- `list[list[float]]`: A nested list containing correlation values for each gene in the chunk compared to all other genes.

> Raises `MethodError` if an unsupported correlation method is provided.


In [ ]:
def _correlate_chunk(args):
    data, method, start, end = args  
    result = []
    for i in range(start, end):  
        row_i = data[i]
        row_corr = []
        for j in range(len(data)):
            if method == 'pearson':
                corr = np.corrcoef(row_i, data[j])[0, 1]
            elif method == 'spearman':
                corr = pd.Series(row_i).corr(pd.Series(data[j]), method='spearman')
            elif method == 'kendall':
                corr = pd.Series(row_i).corr(pd.Series(data[j]), method='kendall')
            else:
                raise MethodError(method)
            row_corr.append(corr)
        result.append(row_corr)
    return result

def parallel_correlation(df: pd.DataFrame, method: str, n_processes: int):
    df = df.set_index(df.columns[0])
    data = df.to_numpy()
    n_genes = data.shape[0]
    chunk_size = max(1, n_genes // (n_processes or mp.cpu_count()))

    args = [
        (data, method, i, min(i + chunk_size, n_genes))
        for i in range(0, n_genes, chunk_size)
    ]
    
    with mp.Pool(processes=n_processes or mp.cpu_count()) as pool:
        chunks = pool.map(_correlate_chunk, args)
    
    corr_matrix = np.vstack(chunks)
    corr_df = pd.DataFrame(corr_matrix, index=df.index, columns=df.index)
    corr_df = corr_df.round(2).dropna(axis=0, how='all').dropna(axis=1, how='all')
    return corr_df

### Correlation Heatmap

An interactive and explorative heatmap is returned by this function and saved as `html` file.

**Parameters**
- `correlation_matrix` (`pl.DataFrame`): A polars DataFrame where each rows and columns represents a gene and the cells contain the correlation between genes .
- `basename_obj` (`AbsolutePath`): is used to build the name of the output dir and the other results
- `output_dir`: as above, is used to store the results in a dedicated directory

**Returns**
- Saves an interactive and explorative heatmap in the output folder.

In [ ]:
def correlation_heatmap(correlation_matrix: pd.DataFrame, basename_obj: AbsolutePath, output_dir):
    print('drawing an heatmap...')

    fig = px.imshow(correlation_matrix.values, 
                    labels=dict(x="Genes", y="Genes", color="Correlation"),
                    x=correlation_matrix.columns, 
                    y=correlation_matrix.columns,
                    color_continuous_scale='RdBu_r',
                    aspect="auto")
    
    fig.update_layout(
        title="Correlation Heatmap",
        xaxis=dict(tickangle=90, tickfont=dict(size=8)),
        yaxis=dict(tickfont=dict(size=8)),
        autosize=True,
        height=800,) 

    fig.write_html(os.path.join(output_dir, f'{basename_obj.basename}_correlationHeatmap.html'))

    print(f'Interactive heatmap saved.')

### Dendrogram

An interactive and explorative dendrogram is returned by this function and saved as `html` file.

**Parameters**
- `correlation_matrix` (`pl.DataFrame`): A polars DataFrame where each rows and columns represents a gene and the cells contain the correlation between genes .
- `basename_obj` (`AbsolutePath`): is used to build the name of the output dir and the other results
- `output_dir`: as above, is used to store the results in a dedicated directory

**Returns**
- Saves an interactive and explorative dendrogram in the output folder.

In [ ]:
def dendrogram_plot(correlation_matrix, basename_obj: AbsolutePath, output_dir):
    print('drawing a dendrogram...')
    distance_matrix = 1 - correlation_matrix
    distance_matrix = (distance_matrix + distance_matrix.T) / 2  #symmetry
    np.fill_diagonal(distance_matrix.values, 0)  # ensure zero diagonal
    condensed = squareform(distance_matrix.values)
    Z = linkage(condensed, method='average')

    fig = ff.create_dendrogram(
        distance_matrix.values,
        labels=correlation_matrix.columns.tolist(),
        linkagefun=lambda _: Z  # Already computed
    )

    fig.update_layout(
        title="Gene Dendrogram",
        xaxis=dict(tickangle=90, tickfont=dict(size=8)),
        yaxis=dict(tickfont=dict(size=8)),
        autosize=True,
        height=800,
    )

    fig.write_html(os.path.join(output_dir, f'{basename_obj.basename}_dendrogram.html'))
    print('dendrogram saved.')

### building the network

This function builds an undirected gene co-expression network based on a correlation matrix. Genes are added as nodes, and edges are added between pairs of genes whose correlation exceeds a specified threshold.

#### **Parameters**
- `correlation_matrix` (`pd.DataFrame`): A symmetric DataFrame where rows and columns represent genes, and values are correlation coefficients between gene expression profiles.
- `corr_threshold` (`float`): Minimum correlation value required to create an edge between two genes, is an optional parameter.

#### **Behavior**
1. Initializes an empty undirected graph using `networkx`.
2. Adds each gene (column name) as a node in the graph.
3. Iterates through the upper triangle of the correlation matrix (excluding the diagonal).
4. For each gene pair `(i, j)`, adds an edge with weight equal to the correlation coefficient if it meets or exceeds the specified threshold.
5. If no edges are created (i.e., no pair of genes meets the threshold), prints an error message and returns `None`.

#### **Returns**
- `networkx.Graph`: A graph where nodes represent genes and edges represent high correlation relationships between genes.
  - If no edges meet the threshold, returns `None`.

>  Useful for conceptualizing co-expression networks or for downstream network analysis.


In [ ]:
def build_network(correlation_matrix: pd.DataFrame, corr_threshold: float):
    G = nx.Graph()
    G.add_nodes_from(correlation_matrix.columns)
    
    for i in range(len(correlation_matrix)):
        for j in range(i + 1, len(correlation_matrix)):
            gene1 = correlation_matrix.index[i]
            gene2 = correlation_matrix.columns[j]
            correlation_value = correlation_matrix.iloc[i, j]
            if correlation_value >= corr_threshold:
                G.add_edge(gene1, gene2, weight=correlation_value)

    if G.number_of_edges() == 0:
        print('ERROR, no edges detected')
        return
    return G

### Conducting the centrality analysis

This function performs network centrality analysis on a given network, computing several topological metrics for each node. The results are saved to a `.txt` file in the specified output directory.

### Parameters

- `network` (`nx.Graph`):  
  A NetworkX graph object representing the network to analyze.  
  It should contain at least one connected component.

- `basename_obj` (`AbsolutePath`):  
  An object with a `.basename` attribute used to name the output file.

- `output_dir` (`str`):  
  Path to the directory where the output file will be saved.

### Computed Metrics

- **Degree Centrality**  
  Measures the number of direct connections a node has.

- **Closeness Centrality**  
  Indicates how close a node is to all other nodes in the network.

- **Eigenvector Centrality**  
  Measures the influence of a node, taking into account the importance of its neighbors.  
  Computed only on the largest connected component of the graph.

- **Clustering Coefficient**  
  Measures the degree to which nodes tend to cluster together.

### Output
Creates a text file named:

```
{basename}_centrality_analysis.txt
```

> Each section of the file lists the nodes and their corresponding centrality values.

In [ ]:
def centrality_analysis(network: nx.Graph, basename_obj: AbsolutePath, output_dir):
    largest_cc = max(nx.connected_components(network), key=len)
    subgraph = network.subgraph(largest_cc)
    
    degree_centrality = nx.degree_centrality(network)
    closeness_centrality = nx.closeness_centrality(network)
    eigen_centrality = nx.eigenvector_centrality(subgraph, max_iter=1000)
    clustering = nx.clustering(network)
    
    with open(os.path.join(output_dir, f'{basename_obj.basename}_centrality_analysis.txt'), 'w') as file:
        file.write("Degree Centrality:\n")
        for gene, centrality in degree_centrality.items():
            file.write(f"{gene}: {centrality:.2f}\n")
        file.write("\nCloseness Centrality:\n")
        for gene, centrality in closeness_centrality.items():
            file.write(f"{gene}: {centrality:.2f}\n")
        file.write("\nEigenvector Centrality:\n")
        for gene, centrality in eigen_centrality.items():
            file.write(f"{gene}: {centrality:.2f}\n")
        file.write("\nClustering Coefficient:\n")
        for gene, coeff in clustering.items():
            file.write(f"{gene}: {coeff:.2f}\n")
    
    return degree_centrality, closeness_centrality, eigen_centrality, clustering

## Graph Analysis Functions for Cytoscape and GO Enrichment

Saves a NetworkX graph in `.graphml` format, enriched with node attributes for centrality metrics. This file can be directly imported into **Cytoscape** for visualization and further analysis.

#### Parameters
- **`network`**: NetworkX graph object with nodes representing genes.
- **`basename_obj`**: Object containing `.basename` used to name the output file.
- **`output_dir`**: Directory where the `.graphml` file will be saved.
- **`degree_centrality`**, **`closeness_centrality`**, **`eigen_centrality`**, **`clustering`**: Dictionaries mapping gene names to respective centrality metrics.

#### Output
Creates a file containing the full network with node attributes.


In [ ]:
def save_cytoscape_format(
    network: nx.Graph,
    basename_obj: AbsolutePath,
    output_dir: str,
    degree_centrality: dict,
    closeness_centrality: dict,
    eigen_centrality: dict,
    clustering: dict):
    print('Saving file for Cytoscape analysis...')

    for node in network.nodes():
        network.nodes[node]['degree_centrality'] = round(degree_centrality.get(node, 0), 2)
        network.nodes[node]['closeness_centrality'] = round(closeness_centrality.get(node, 0), 2)
        network.nodes[node]['eigen_centrality'] = round(eigen_centrality.get(node, 0), 2)
        network.nodes[node]['clustering_coefficient'] = round(clustering.get(node, 0), 2)

    nx.write_graphml(network, os.path.join(output_dir, f'{basename_obj.basename}.graphml'))
    print(f'Graphml file saved at wdir')

### Getting gene cluster

Detects gene clusters (communities) in the input graph using the **Louvain algorithm** for community detection.

#### Parameters
- **`G`**: A NetworkX graph.

#### Returns
- A list of clusters, where each cluster is a list of gene names (nodes).

In [ ]:
def get_gene_clusters(G):
    return [list(c) for c in community.louvain_communities(G)]


### GO Enrichment

Performs **GO Biological Process enrichment** using the Enrichr API for a single cluster of genes.

#### Parameters
- `gene_list`: List of genes (node names) from one cluster.
- `output_dir`: Directory where the enrichment results will be saved.
- `cluster_id`: Numeric ID of the cluster (used in folder naming).

#### Notes
- Uses the gene set: `GO_Biological_Process_2025`.
- only results with `adjusted p-value < 0.1` are returned.
- handles errors and prints warnings if enrichment fails.

In [ ]:
def run_go_enrichment(gene_list, output_dir, cluster_id):
    try:
        gp.enrichr(
            gene_list=gene_list,
            gene_sets='GO_Biological_Process_2025',
            outdir=os.path.join(output_dir, f'cluster_{cluster_id}_GO'),
            cutoff=0.1
        )
    except Exception as i:
        print(f'[GO] Enrichment failed for cluster {cluster_id}: {i}')

### Running GO for all cluster

Applies GO enrichment (`run_go_enrichment`) to **all clusters** detected in the graph using Louvain community detection.

#### Parameters
- **`G`**: A NetworkX graph representing the gene network.
- `output_dir`: Directory where all enrichment results will be saved, with subfolders for each cluster.

#### Workflow
1. Detect communities (clusters) in the graph.
2. For each cluster:
   - Run GO enrichment.
   - Save results to `output_dir/cluster_{i}_GO`.

In [ ]:
def go_enrichment_all_clusters(G, output_dir):
    clusters = get_gene_clusters(G)
    for i, gene_list in enumerate(clusters):
        run_go_enrichment(gene_list, output_dir, i)

### Parser

List of commands, necessary and optional (the ones reported as: --command_name), all with a default value for the analysis.

In [ ]:
def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("expression_file", type=str, 
                        help="Path to gene expression matrix (.csv or .tsv)")
    parser.add_argument("--expression_threshold", type=float, default=1.0,
                        help='Expression level threshold for filtering genes')
    parser.add_argument("--correlation_threshold", type=float, default=0.85,
                        help='Correlation threshold to create edges in network')
    parser.add_argument("--method", type=str, choices=["pearson", "spearman", "kendall"], default="pearson",
                        help='Correlation method (pearson, spearman or kendall)')
    parser.add_argument("--top_genes", type=int, default=1000,
                        help='choose the chunck size and the top genes desired')
    parser.add_argument("--go_enrichment", type=str, choices=['yes','y','n','no'], default='no',
                        help='choose if you want to perform GO enrichment analysis')
    parser.add_argument("--cytoscape_file", type=str, choices=['yes', 'y', 'no', 'n'] , default='yes',
                        help='y or n for saving a file for external cytoscape analysis')
    parser.add_argument("--n_processes", type=int, default=4, 
                        help="Number of processes to use to calculate the correlation")
    return parser.parse_args()

### Main body of the program

In [ ]:
def main():
    #parser
    args = get_args()
    
    #path for the esperiment
    basename = AbsolutePath(os.path.splitext(os.path.basename(args.expression_file))[0])
    output_folder = basename.output_dir

    print('starting analysis...')

    #write dataframe into csv
    expression_dataset = load_expression_data(args.expression_file, basename, output_folder)
    expression_dataset = remove_least_expressed_genes(expression_dataset, args.expression_threshold)
    expression_dataset = subset_high_variance_genes(expression_dataset, args.top_genes)
    expression_dataset.write_csv(os.path.join(output_folder, f'{basename.basename}_filtered_expression.csv'))

    #correlation matrix
    correlation_matrix = parallel_correlation(expression_dataset.to_pandas(), 
                                              method=args.method, n_processes=args.n_processes).round(2)
    correlation_matrix.to_csv(os.path.join(output_folder, f'{basename.basename}_correlation_matrix.csv'), 
                              float_format="%.2f")
    ##Check
    if correlation_matrix.isnull().all().all(): #2 all for the 2D 
        raise ValueError("Only NAN")

    #graph
    correlation_heatmap(correlation_matrix, basename, output_folder)
    dendrogram_plot(correlation_matrix, basename, output_folder)

    #network se non è vuoto il tutto 
    network = build_network(correlation_matrix, args.correlation_threshold)
    if network:
        degree, closness, eigen, cluster = centrality_analysis(network, basename, output_folder)

        #cytoscape file
        if args.cytoscape_file.lower() in ['yes', 'y']:
            save_cytoscape_format(network, basename, output_folder, degree, closness, eigen, cluster)


        #GO enrichment
        if args.go_enrichment.lower() in ['yes', 'y']:
            go_enrichment_all_clusters(network, output_folder)

    print('Analysis Finished')

if __name__ == "__main__":
    main()